# Extend EirGrid history and build forecast vintages

This notebook is the documented entry point for performance-backlog Issues #2 and #3. It builds a 2021-present half-hourly core history and a separate long-form forecast-vintage table. The separation matters: observations describe what happened, while vintages record what was known at each prediction time.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
PROCESSED = REPO_ROOT / 'data' / 'processed'

## 1. Reproducible build

The command downloads only missing files. Cached bytes are SHA-256 checked, the EirGrid GMT offset is applied before UTC conversion, and individual SEMO revisions retain target, publication, and retrieval timestamps.

In [ ]:
subprocess.run(
    [sys.executable, str(REPO_ROOT / 'scripts' / 'build_extended_data.py'), '--forecast-days', '3'],
    cwd=REPO_ROOT,
    check=True,
)

## 2. Inspect the clean outputs

Gzip keeps the repository manageable; pandas detects it automatically. Missing optional measurements are intentional and paired with availability flags rather than invented values.

In [ ]:
history = pd.read_csv(PROCESSED / 'eirgrid_core_history_30min.csv.gz', parse_dates=['timestamp_utc'])
vintages = pd.read_csv(
    PROCESSED / 'forecast_vintages.csv.gz',
    parse_dates=['target_timestamp_utc', 'target_end_timestamp_utc', 'published_at_utc', 'downloaded_at_utc'],
)
asof_features = pd.read_csv(
    PROCESSED / 'forecast_features_asof_30_60.csv',
    parse_dates=['issue_timestamp_utc', 'target_timestamp_utc'],
)
display(history.head(3))
display(vintages.head(3))
display(asof_features.head(3))

## 3. Quality and leakage gates

Every selected forecast publication must be at or before its model issue time. The JSON reports also expose exact field/source coverage and publication-lag distributions.

In [ ]:
history_quality = json.loads((PROCESSED / 'eirgrid_history_quality_report.json').read_text())
forecast_quality = json.loads((PROCESSED / 'forecast_vintage_quality_report.json').read_text())
asof_quality = json.loads((PROCESSED / 'forecast_asof_quality_report.json').read_text())

assert history_quality['duplicate_natural_keys'] == 0
assert history_quality['max_dispatch_accounting_difference_mwh'] < 0.05
assert history_quality['meets_24_month_target']
assert forecast_quality['duplicate_vintage_keys'] == 0
assert asof_quality['future_publication_violations'] == 0

display(pd.DataFrame(forecast_quality['feature_coverage']).T)
display(pd.Series(asof_quality['availability_rate_by_feature'], name='availability_rate'))

## How this feeds modelling

Issue #4 can join engineered lag/ramp/weather features to `eirgrid_core_history_30min.csv.gz`. Forecast features must come from `forecast_features_asof_30_60.csv`, never from a latest-forecast overwrite. The current live forecast window is newer than the latest monthly dispatch-down label, so its label availability is correctly zero until EirGrid publishes that report.